# From Raw X300 IQ to LTE DL/UL PCAP

## Experiment background

This notebook analyzes an authorized, passive recording of a real-world **T-Mobile / Starlink Direct-to-Cell LTE** experiment. We used an Ettus **USRP X300 software-defined radio (SDR)** as a nearby third-party receiver. The X300 did not transmit or join the network; it recorded the radio waveform for offline research analysis.

Two X300 receive channels started from the same hardware time:

| X300 channel | Link direction | Center frequency | LTE channel bandwidth | Recording rate |
|---|---|---:|---:|---:|
| Channel 0 | Satellite/base-station downlink (DL) | 1992.5 MHz | 5 MHz, 25 resource blocks | 10 MS/s |
| Channel 1 | Phone/UE uplink (UL) | 1912.5 MHz | 5 MHz, 25 resource blocks | 10 MS/s |

The 5 MHz value is the nominal occupied LTE channel bandwidth. The X300 stored 10 million complex I/Q samples per second on each channel, giving extra frequency space around the LTE carrier. This tutorial uses a synchronized **five-second excerpt** (original capture time 4.000–9.000 seconds), containing 50,000,000 aligned samples per channel in little-endian `fc32` format. The complete 150-second source capture remains in the experiment archive; it is intentionally excluded from this teaching repository.

This interval was selected because it contains a compact end-to-end exchange on PCI 427: MIB and SIB broadcasts, RRC connection setup, identity request/response, attach signaling, authentication request/response, security-mode signaling, timing-advance commands, and many CRC-valid UL transport blocks.

The research goal is to start from these two raw waveform files and reconstruct:

- how a satellite-beam LTE cell and its PCI are identified;
- broadcast and dedicated downlink messages, including MIB and SIB information;
- the downlink DCI/RAR grants that scheduled phone transmissions;
- the corresponding uplink PUSCH bursts and CRC-valid payloads;
- a combined DL/UL packet trace that can be inspected in Wireshark.

This is an executable, restartable pipeline. It is written for computer science undergraduates, so radio concepts are introduced when they become necessary.

The notebook performs the complete sequence:

1. validate and downsample the five-second DL and UL IQ files;
2. align and decode PCI 427, including MIB/SIB/RRC and DCI/RAR grants;
3. use those DL grants and RRC parameters to acquire and decode UL PUSCH;
4. keep CRC-valid UL transport blocks;
5. create and store the per-PCI and combined PCAP/PCAPNG files;
6. visualize the evidence at each stage.

Every expensive stage uses **generate-if-missing** behavior. A stored output is validated and reported as `SKIP`; if it is absent or incomplete, the notebook runs the checked-in decoder stage and stores the new result under `Tutorial/decoder_results/`.

The raw IQ files are never modified. Decoder outputs are ignored by Git, while the pipeline code and fixed-capture segment configuration are tracked. The patched LTEsniffer and srsRAN helper binaries are required only when a missing decoder output must be rebuilt. Figures appear inside Jupyter and are not saved as image files.

## Roadmap: raw IQ to PCAP

1. Validate the simultaneous X300 DL and UL files.
2. Downsample both complete traces from 10 MS/s to 5.76 MS/s once.
3. View a one-second processed DL/UL waterfall.
4. Search the DL for LTE synchronization, identify PCIs, and confirm MIBs.
5. Decode DL PCAPs, including SIB1 cell identity and scheduling grants.
6. Use DL DCI0/RAR grants to predict each scheduled UL time and PRB range.
7. Search timing/CFO candidates, verify PUSCH DMRS, apply RRC/UCI parameters, and test the transport-block CRC.
8. Keep only CRC-valid UL payloads and combine MIB, DL, and UL into PCAP/PCAPNG.

## 1. Import the Python libraries and find the data

The raw-data directory is next to `Tutorial/`. An optional `LTE_RAW_DATA_DIR` environment variable is also supported for testing or a different directory layout.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import matplotlib.pyplot as plt
import subprocess

plt.rcParams.update({"figure.figsize": (12, 5)})

candidates = ([Path(os.environ["LTE_RAW_DATA_DIR"])] if "LTE_RAW_DATA_DIR" in os.environ else []) + [
    Path("raw_data"),
    Path("Tutorial/raw_data"),
]
RAW_DIR = next((path.resolve() for path in candidates if path.is_dir()), None)
if RAW_DIR is None:
    raise FileNotFoundError(
        "Cannot find Tutorial/raw_data. Start Jupyter from "
        "the repository root or the Tutorial directory."
    )

META_PATH = RAW_DIR / "capture.json"
DL_PATH = RAW_DIR / "downlink.fc32"
UL_PATH = RAW_DIR / "uplink.fc32"
REPO_ROOT = RAW_DIR.parents[1]
TUTORIAL_DIR = REPO_ROOT / "Tutorial"
LOCAL_PROCESSED_DIR = (TUTORIAL_DIR / "processed_data").resolve()
LOCAL_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_OVERRIDE_PATH = LOCAL_PROCESSED_DIR / "source_override.json"
processed_override = (
    json.loads(SOURCE_OVERRIDE_PATH.read_text())
    if SOURCE_OVERRIDE_PATH.is_file() else {}
)
selected_processed = os.environ.get("LTE_PROCESSED_DATA_DIR") or processed_override.get("directory")
PROCESSED_DIR = Path(selected_processed or LOCAL_PROCESSED_DIR).expanduser().resolve()
DL_PROCESSED_PATH = PROCESSED_DIR / processed_override.get(
    "downlink_file", "downlink_5p76Msps.fc32"
)
UL_PROCESSED_PATH = PROCESSED_DIR / processed_override.get(
    "uplink_file", "uplink_5p76Msps.fc32"
)

metadata = json.loads(META_PATH.read_text())
capture = metadata["capture"]
print("Raw data directory:", RAW_DIR)
print(json.dumps(metadata, indent=2))

## 2. What is an IQ sample?

The X300 does not directly store packets. It stores measurements of the radio waveform.

Each measurement contains two numbers:

- **I**: the real part of the sample;
- **Q**: the imaginary part of the sample.

Python writes one sample as:

```python
sample = I + 1j * Q
```

The file type is `fc32`: I and Q are each 32-bit floating-point numbers. One complex sample therefore occupies 8 bytes.

The downlink and uplink files started at the same hardware time. Sample number `n` in the DL file occurred at the same time as sample number `n` in the UL file. We need to preserve this relationship later when matching a downlink scheduling message with the corresponding uplink transmission.

In [ ]:
FC32 = np.dtype("<c8")       # little-endian complex64: float32 I + float32 Q
BYTES_PER_SAMPLE = FC32.itemsize
INPUT_RATE_HZ = int(capture["sample_rate_hz"])
EXPECTED_SAMPLES = int(capture["sample_count_per_channel"])

def describe_file(path: Path) -> dict:
    byte_count = path.stat().st_size
    if byte_count % BYTES_PER_SAMPLE:
        raise ValueError(f"{path.name} does not contain complete fc32 samples")
    sample_count = byte_count // BYTES_PER_SAMPLE
    return {
        "name": path.name,
        "size_GB": byte_count / 1e9,
        "samples": sample_count,
        "duration_seconds": sample_count / INPUT_RATE_HZ,
    }

dl_info = describe_file(DL_PATH)
ul_info = describe_file(UL_PATH)
print(dl_info)
print(ul_info)

assert dl_info["samples"] == ul_info["samples"]
assert dl_info["samples"] == EXPECTED_SAMPLES
print(f"\nBoth channels are aligned and contain {dl_info['duration_seconds']:.3f} seconds.")

## 3. Open the large files without loading them into memory

Each file is 12.08 GB. `np.memmap` lets Python treat a file like an array while reading only the small section that we request.

In [ ]:
dl_iq = np.memmap(DL_PATH, dtype=FC32, mode="r")
ul_iq = np.memmap(UL_PATH, dtype=FC32, mode="r")

print("DL array shape:", dl_iq.shape)
print("UL array shape:", ul_iq.shape)
print("First five DL samples:", dl_iq[:5])
print("First five UL samples:", ul_iq[:5])

## 4. Create the continuous 5.76 MS/s decoder inputs once

The 5 MHz LTE decoder expects 5.76 million samples per second. The DL and UL
must pass through one continuous, identical rational-resampler chain. Filtering
independent one-second blocks creates small phase discontinuities that can make
LTE synchronization fail much later in the file even though the output size
looks correct.

The repository setup installs GNU Radio. This cell uses its streaming rational
resampler for both channels in one flowgraph. A versioned manifest distinguishes
the continuous output from older blockwise files. Valid existing outputs print
`SKIP`; otherwise new files are completed under temporary names and atomically
replace the old generated files.

In [ ]:
DECODER_RATE_HZ = 5_760_000
NOMINAL_DECODER_SAMPLES = EXPECTED_SAMPLES * DECODER_RATE_HZ // INPUT_RATE_HZ
PROCESSING_MANIFEST = (
    SOURCE_OVERRIDE_PATH if processed_override else PROCESSED_DIR / "processing.json"
)
RESAMPLER = TUTORIAL_DIR / "utils" / "decoder_core" / "scripts" / "channelize_pair.py"

def continuous_pair_is_complete() -> bool:
    if not all(path.is_file() for path in (
        DL_PROCESSED_PATH, UL_PROCESSED_PATH, PROCESSING_MANIFEST
    )):
        return False
    try:
        saved = json.loads(PROCESSING_MANIFEST.read_text())
    except (OSError, json.JSONDecodeError):
        return False
    dl_samples = DL_PROCESSED_PATH.stat().st_size // BYTES_PER_SAMPLE
    ul_samples = UL_PROCESSED_PATH.stat().st_size // BYTES_PER_SAMPLE
    return (
        saved.get("method") == "gnuradio.rational_resampler_ccc"
        and saved.get("input_sample_rate_hz") == INPUT_RATE_HZ
        and saved.get("output_sample_rate_hz") == DECODER_RATE_HZ
        and dl_samples == ul_samples
        and abs(dl_samples - NOMINAL_DECODER_SAMPLES) <= 256
    )

if continuous_pair_is_complete():
    print("SKIP: validated continuous 5.76 MS/s DL/UL pair")
else:
    if processed_override:
        raise RuntimeError(
            f"The processed-IQ override is incomplete or invalid: {SOURCE_OVERRIDE_PATH}"
        )
    gnuradio_check = subprocess.run(
        ["/usr/bin/python3", "-c", "import gnuradio"], check=False
    )
    if gnuradio_check.returncode:
        raise RuntimeError(
            "GNU Radio is missing. From the repository root run "
            "`bash Tutorial/setup_ubuntu.sh`, restart the kernel, and retry."
        )
    dl_partial = DL_PROCESSED_PATH.with_suffix(".fc32.partial")
    ul_partial = UL_PROCESSED_PATH.with_suffix(".fc32.partial")
    manifest_partial = PROCESSED_DIR / "processing.partial.json"
    for partial in (dl_partial, ul_partial, manifest_partial):
        partial.unlink(missing_ok=True)
    subprocess.run([
        "/usr/bin/python3", str(RESAMPLER), "--manifest", str(META_PATH),
        "--dl-output", str(dl_partial), "--ul-output", str(ul_partial),
        "--output-rate", str(DECODER_RATE_HZ), "--result", str(manifest_partial),
    ], check=True)
    dl_samples = dl_partial.stat().st_size // BYTES_PER_SAMPLE
    ul_samples = ul_partial.stat().st_size // BYTES_PER_SAMPLE
    if dl_samples != ul_samples or abs(dl_samples - NOMINAL_DECODER_SAMPLES) > 256:
        raise RuntimeError(
            f"Invalid continuous resampler output: DL={dl_samples:,}, UL={ul_samples:,}"
        )
    dl_partial.replace(DL_PROCESSED_PATH)
    ul_partial.replace(UL_PROCESSED_PATH)
    processing = json.loads(manifest_partial.read_text())
    processing.update({
        "source_samples_per_channel": EXPECTED_SAMPLES,
        "processed_samples_per_channel": dl_samples,
        "downlink_file": DL_PROCESSED_PATH.name,
        "uplink_file": UL_PROCESSED_PATH.name,
    })
    PROCESSING_MANIFEST.write_text(json.dumps(processing, indent=2) + "\n")
    manifest_partial.unlink(missing_ok=True)
    print(f"CREATE: {dl_samples:,} continuous samples per channel")

processing = json.loads(PROCESSING_MANIFEST.read_text())
EXPECTED_DECODER_SAMPLES = int(processing["processed_samples_per_channel"])
print("Processed data directory:", PROCESSED_DIR)
print(json.dumps(processing, indent=2))

## 5. Open the processed DL and UL files

These arrays cover the complete recording at the decoder's input rate. They remain sample-aligned: sample `n` in processed DL corresponds to sample `n` in processed UL.

In [ ]:
decoder_dl = np.memmap(DL_PROCESSED_PATH, dtype=FC32, mode="r")
decoder_ul = np.memmap(UL_PROCESSED_PATH, dtype=FC32, mode="r")

print("Processed DL shape:", decoder_dl.shape)
print("Processed UL shape:", decoder_ul.shape)
assert decoder_dl.size == decoder_ul.size == EXPECTED_DECODER_SAMPLES

## 6. Display one second of processed downlink and uplink spectrum

A normal line plot is not very helpful for millions of radio samples. Instead, we repeatedly calculate a short frequency spectrum and stack the results vertically. This is called a **waterfall**:

- the horizontal axis is frequency;
- the vertical axis is time;
- brighter colors mean stronger received signals.

The demo reads its one-second interval directly from the reusable processed files. The function samples a few hundred short windows, so it remains fast and memory-efficient.

In [ ]:
def one_second_waterfall(
    path: Path,
    sample_rate_hz: float,
    start_s: float,
    *,
    fft_size: int = 4096,
    rows: int = 500,
):
    source = np.memmap(path, dtype=FC32, mode="r")
    first = int(round(start_s * sample_rate_hz))
    last = min(source.size, first + int(sample_rate_hz))
    if first < 0 or last - first < fft_size:
        raise ValueError("The selected second is outside the recording")

    starts = np.linspace(first, last - fft_size, rows, dtype=np.int64)
    window = np.hanning(fft_size).astype(np.float32)
    result = np.empty((rows, fft_size), dtype=np.float32)

    for row, offset in enumerate(starts):
        samples = np.asarray(source[offset:offset + fft_size])
        spectrum = np.fft.fftshift(np.fft.fft(samples * window))
        result[row] = 10 * np.log10(np.abs(spectrum) ** 2 + 1e-20)

    # Relative power makes strong and weak structures easier to compare visually.
    result -= np.percentile(result, 99.8)
    times = starts / sample_rate_hz
    frequencies = np.fft.fftshift(np.fft.fftfreq(fft_size, 1 / sample_rate_hz))
    return times, frequencies, result


SPECTRUM_START_S = 0.4
center_frequencies_hz = {
    "Downlink": float(capture["dl_frequency_hz"]),
    "Uplink": float(capture["ul_frequency_hz"]),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for axis, (name, path) in zip(
    axes,
    (("Downlink", DL_PROCESSED_PATH), ("Uplink", UL_PROCESSED_PATH)),
):
    times, offsets, power_db = one_second_waterfall(path, DECODER_RATE_HZ, SPECTRUM_START_S)
    frequencies_mhz = (center_frequencies_hz[name] + offsets) / 1e6
    image = axis.imshow(
        power_db,
        origin="upper",
        aspect="auto",
        extent=[frequencies_mhz[0], frequencies_mhz[-1], times[-1], times[0]],
        cmap="turbo",
        vmin=-55,
        vmax=0,
    )
    axis.set_title(f"{name} (processed at 5.76 MS/s)")
    axis.set_xlabel("Frequency (MHz)")

axes[0].set_ylabel("Capture time (seconds)")
fig.colorbar(image, ax=axes, label="Relative power (dB)", shrink=0.85)
fig.suptitle(f"One second of simultaneous DL and UL: {SPECTRUM_START_S:.1f}–{SPECTRUM_START_S + 1:.1f} s")
plt.show()

## 7. What we have before decoding

At this point:

- both complete recordings exist at the decoder's 5.76 MS/s rate;
- `decoder_dl` and `decoder_ul` access them without loading 13.9 GB into RAM;
- DL sample `n` and UL sample `n` still refer to the same receiver time;
- no visual feature has yet been called a packet.

The remaining sections move from visual energy to validated LTE structures. Synchronization, MIB, DMRS, and CRC checks—not the appearance of the waterfall—are what make a decoding claim trustworthy.

## 8. Generate or reuse all downlink results first

The notebook now calls a persistent pipeline from `Tutorial/utils/pipeline.py`. This stage finishes every configured PCI's downlink before any expensive uplink search. For an existing complete DL result, it prints `SKIP`. If something is missing, it invokes the repository-local decoder implementation under `Tutorial/utils/decoder_core/`:

- DL cell search, MIB confirmation, alignment, and LTEsniffer replay;
- measured per-segment CFO seeds from the fixed-capture configuration, followed by an exact aligned-duration check;
- DCI/RAR grant parsing and RRC parameter extraction.

A clean-machine setup is provided by `Tutorial/setup_ubuntu.sh`. DL processing is normally much shorter than the later wide UL search, so the DL tables and plots become available first.

The pipeline enables LTEsniffer's diagnostic grant records during UL-mode replay. It stops before the expensive PUSCH search if alignment is short or the DL replay produces no parseable UL grants. After generation, the remaining notebook cells analyze PCI/beam coverage, SIB1 identities, grants, timing/CFO candidates, DMRS acceptance, CRC outcomes, and PCAP contents.

In [ ]:
import importlib
import sys
from IPython.display import HTML, Markdown, display

# This notebook works when Jupyter starts from either the repository root or Tutorial/.
REPO_ROOT = RAW_DIR.parents[1]
if str(REPO_ROOT / "Tutorial") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "Tutorial"))

import utils.pipeline as pipeline_module
import utils.tutorial_tools as tutorial_tools_module

# Jupyter keeps imported Python modules in memory. Reload these local helpers so
# rerunning this cell picks up path or decoder changes made since the kernel began.
importlib.invalidate_caches()
importlib.reload(tutorial_tools_module)
importlib.reload(pipeline_module)

from utils.tutorial_tools import (
    decoder_paths, html_table, load_jsonl, load_manifest,
    pcap_protocol_counts, pcap_summary, plot_cell_waterfall,
    plot_crc_outcomes, plot_timing_candidates, plot_ul_grants,
    summarize_downlink, summarize_uplink,
)
from check_setup import require_setup
TutorialPipeline = pipeline_module.TutorialPipeline

PATHS = decoder_paths(REPO_ROOT)
RESULTS_DIR = PATHS["results"]

pipeline = TutorialPipeline(REPO_ROOT)
print("Processed data directory:", pipeline.processed)
print("Decoder results directory:", pipeline.results)
require_setup(TUTORIAL_DIR, require_gnuradio=False)
segment_outputs, dl_pipeline_report = pipeline.ensure_downlinks()
manifest = load_manifest(RESULTS_DIR)

display(HTML(html_table(dl_pipeline_report)))
print("All configured DL segments are ready for analysis.")

### Why the pipeline stores intermediate files

The pipeline preserves cell/MIB results, each DL PCAP and enhanced grant log, burst boundaries, timing/CFO/DMRS candidates, every transport-block attempt, CRC-valid UL JSONL, and the final PCAPs.

Keeping these files lets the later cells explain failures as well as successes. It also makes the notebook restartable: deleting one stored stage causes that stage to regenerate, while independent valid stages remain untouched.

# Part 2 — Decode the downlink first

The downlink provides the information needed to interpret the uplink. We first locate LTE synchronization signals, identify a **Physical Cell ID (PCI)**, and confirm the result by decoding the **MIB**. Later DL messages provide SIB1/SIB2, RRC configuration, DCI scheduling grants, RAR, and timing-advance commands.

## 9. Coarse cell search and MIB confirmation

LTE synchronization uses PSS/SSS sequences. Correlation supplies a possible PCI, timing, and carrier-frequency offset (CFO). A correlation peak is only a candidate; decoding the PBCH/MIB confirms that the timing and PCI are internally consistent LTE.

The excerpt is centered on PCI 427. The MIB decode confirms that the synchronization peak, timing, and PCI form a consistent LTE downlink rather than an accidental correlation.

In [ ]:
cell_search = json.loads((RESULTS_DIR / "cell_search_results.json").read_text())
search_rows = []
for item in cell_search:
    if item.get("found") and item.get("frame_type") == "FDD":
        search_rows.append({
            "PCI candidate": item["pci"],
            "input time (s)": round(item.get("time_s", item.get("offset_samples", 0) / 1_920_000), 3),
            "CFO (Hz)": round(item["cfo_hz"], 1) if "cfo_hz" in item else "recorded in alignment log",
            "peak/side ratio": round(item["psr"], 2) if "psr" in item else "MIB confirmed",
            "frame type": item["frame_type"],
        })
display(HTML(html_table(search_rows)))

mib_results = json.loads((RESULTS_DIR / "mib_results.json").read_text())
confirmed = [row for row in mib_results if row.get("mib_found")]
print("MIB-confirmed PCIs in the coarse passes:", [row["pci"] for row in confirmed])

## 10. Decode the selected PCI 427 interval

The full recording contains several satellite-beam PCIs, but this compact demonstration intentionally follows one well-populated PCI. The colored box below is the decoder coverage interval after LTE synchronization.

In [ ]:
plot_cell_waterfall(
    DL_PROCESSED_PATH,
    manifest,
    sample_rate_hz=DECODER_RATE_HZ,
    center_hz=float(capture["dl_frequency_hz"]),
)
plt.show()

## 11. SIB1 cell identity: logical satellite/node and sector

SIB1 carries a 28-bit E-UTRAN Cell Identity (ECI). The conventional LTE split is:

```text
ECI = (20-bit eNB/node identifier << 8) | 8-bit sector identifier
```

In this network, the 20-bit value is useful as a **logical satellite/network-node label** and the low 8 bits distinguish a beam/sector. It is not a NORAD spacecraft catalog ID, so the notebook does not claim a physical spacecraft identity from SIB1 alone.

In [ ]:
downlink_rows = summarize_downlink(RESULTS_DIR, manifest)
display(HTML(html_table(downlink_rows)))
display(Markdown(f"**Identity caution:** {manifest['identity_note']}"))

## 12. What the DL PCAPs contain

Each segment's DL-mode PCAP contains MAC-LTE frames that Wireshark can dissect upward into RLC, PDCP, RRC, and NAS when the payload and context permit it. This is where we obtain SIB1/SIB2 and dedicated RRC parameters; DCI0 grant details are also retained by the patched decoder log for the UL stage.

In [ ]:
protocol_rows = []
for cell in manifest["cells"]:
    counts = pcap_protocol_counts(RESULTS_DIR / cell["dl_pcap"])
    protocol_rows.append({
        "segment": cell["label"],
        "PCI": cell["pci"],
        "MAC-LTE": counts.get("mac-lte", 0),
        "RRC": counts.get("lte_rrc", 0),
        "NAS-EPS": counts.get("nas-eps", 0),
    })
display(HTML(html_table(protocol_rows)))

# Part 3 — Use the decoded DL to recover uplink packets

The UL does not transmit the same cell-search signals as the DL. We cannot identify a PUSCH packet merely by finding a bright burst. Instead, the decoder follows this chain:

```text
DL DCI0 or RAR grant
  → expected UL subframe and PRB allocation
  → local timing/CFO candidates in that small region
  → PUSCH DMRS match
  → modulation, TBS, RV, RNTI and RRC/UCI parameters
  → turbo decode
  → transport-block CRC
```

The CRC is the final integrity check. A strong signal and excellent DMRS can still produce wrong payload bits if the coding or UCI hypothesis is wrong.

In [ ]:
# Part 3 is self-initializing: it can be run after a kernel restart without
# depending on Python variables left behind by Parts 1 and 2.
from pathlib import Path
import importlib
import json
import os
import sys

import matplotlib.pyplot as plt
from IPython.display import HTML, display

tutorial_candidates = (
    [Path(os.environ["LTE_TUTORIAL_DIR"])] if "LTE_TUTORIAL_DIR" in os.environ else []
) + [Path.cwd(), Path.cwd() / "Tutorial"]
TUTORIAL_DIR = next(
    (path.resolve() for path in tutorial_candidates if (path / "raw_data/capture.json").is_file()),
    None,
)
if TUTORIAL_DIR is None:
    raise FileNotFoundError(
        "Cannot find Tutorial/raw_data/capture.json. Start Jupyter from the repository "
        "root or Tutorial directory, or set LTE_TUTORIAL_DIR."
    )
REPO_ROOT = TUTORIAL_DIR.parent
RAW_DIR = TUTORIAL_DIR / "raw_data"
PROCESSED_DIR = TUTORIAL_DIR / "processed_data"
RESULTS_DIR = TUTORIAL_DIR / "decoder_results"
DL_PROCESSED_PATH = PROCESSED_DIR / "downlink_5p76Msps.fc32"
UL_PROCESSED_PATH = PROCESSED_DIR / "uplink_5p76Msps.fc32"
DECODER_RATE_HZ = 5_760_000
capture = json.loads((RAW_DIR / "capture.json").read_text())["capture"]

if str(TUTORIAL_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_DIR))
import utils.pipeline as pipeline_module
import utils.tutorial_tools as tutorial_tools_module
importlib.invalidate_caches()
importlib.reload(pipeline_module)
importlib.reload(tutorial_tools_module)
from utils.pipeline import TutorialPipeline
from utils.tutorial_tools import (
    html_table, load_jsonl, load_manifest, pcap_protocol_counts, pcap_summary,
    plot_crc_outcomes, plot_timing_candidates, plot_ul_grants, summarize_uplink,
)
from check_setup import require_setup

require_setup(TUTORIAL_DIR, require_gnuradio=False)
pipeline = TutorialPipeline(REPO_ROOT)
segment_outputs, _ = pipeline.ensure_downlinks()
manifest = load_manifest(RESULTS_DIR)

# This is the expensive stage only when validated UL outputs are absent.
final_pcap, pipeline_report = pipeline.ensure_uplinks(segment_outputs)
display(HTML(html_table(pipeline_report)))
print("Final stored PCAPNG:", final_pcap)

## 13. Read DCI0 uplink grants decoded from the downlink

DCI format 0 is an LTE downlink control message that tells a UE how to transmit PUSCH. Important fields include the UE's RNTI, target subframe, starting PRB, number of PRBs, MCS, transport-block size, redundancy version, DMRS cyclic shift, and whether CQI is requested.

The table shows real grants for PCI 427. `burst_sf` is the aligned-file subframe where we expect UL energy; `prb0` and `len_prb` define the cyan frequency box used in the next plot.

In [ ]:
selected_cell = next(cell for cell in manifest["cells"] if cell["label"] == "pci427_demo")
selected_ul_dir = RESULTS_DIR / "uplink" / selected_cell["ul_result"]
acquisitions = load_jsonl(selected_ul_dir / "grant_directed_acquisition.jsonl")

grant_rows = []
for item in acquisitions[:10]:
    grant = item["grant"]
    grant_rows.append({
        "burst_sf": item["burst"]["burst_sf"],
        "RNTI": grant["rnti"],
        "PRB start": grant["prb_tilde0"],
        "PRB count": grant["len_prb"],
        "MCS": grant["mcs"],
        "TBS bits": grant["tbs"],
        "mod code": grant["mod"],
        "RAR": bool(grant["rar"]),
    })
display(HTML(html_table(grant_rows)))

## 14. Compare scheduled PRBs with the observed UL energy

Each cyan rectangle comes from a DL-decoded grant; it spans the scheduled UL PRBs for one 1 ms LTE subframe. The background is read directly from the full processed UL IQ.

Agreement between a rectangle and a bright burst is useful evidence that we matched the correct region, but it still is not a decoded packet. Several UEs or control signals can occupy nearby PRBs, and the receiver observes a large timing displacement relative to the base-station-side schedule.

In [ ]:
plot_ul_grants(
    UL_PROCESSED_PATH,
    acquisitions,
    selected_cell,
    start_s=0.45,
    duration_s=0.75,
    sample_rate_hz=DECODER_RATE_HZ,
    center_hz=float(capture["ul_frequency_hz"]),
)
plt.show()

## 15. Search timing and CFO, then verify PUSCH DMRS

The receiver is close to the UE, not at the satellite. The UE transmits according to timing advance so its waveform arrives correctly at the satellite, but our nearby receiver sees that transmission early. For PCI 427, the strongest correct family is around −400 µs relative to the uncorrected grant time.

For each grant, the probe searches a timing window and CFO aliases, measures cyclic-prefix consistency, and checks the expected PUSCH DMRS sequence. We retain the best four timing/CFO candidates because the highest acquisition score alone was often a timing alias. Green points below are timing candidates that ultimately produced at least one CRC-valid block.

In [ ]:
crc_valid_rows = load_jsonl(selected_ul_dir / "crc_valid_ul.jsonl")
plot_timing_candidates(acquisitions, crc_valid_rows)
plt.show()

## 16. Apply coding and RRC/UCI parameters, then test CRC

After DMRS acquisition, the decoder still needs the DCI/RAR coding fields and per-UE RRC configuration. In particular, ACK/CQI/RI multiplexing changes the PUSCH bit layout. The Tsat trace commonly used beta-offset indices `(7, 7, 1)` rather than a terrestrial-oriented default.

The decoder therefore prefers time-correct RRC settings extracted from DL PCAPs and uses bounded hypotheses only when the RRC message is unavailable. It tries the plausible candidates and promotes only a nonzero payload whose turbo/transport-block CRC passes.

In [ ]:
uplink_rows = summarize_uplink(RESULTS_DIR, manifest)
display(HTML(html_table(uplink_rows)))
print("Total CRC-valid UL packets:", sum(row["CRC passed"] for row in uplink_rows))
plot_crc_outcomes(uplink_rows)
plt.show()

### Inspect one CRC-valid UL result

The JSONL record preserves the complete audit trail: grant, selected DMRS timing/CFO, UCI hypothesis, modulation, transport-block size, CRC result, and decoded payload. We show only a short payload prefix here; Wireshark should be used for protocol interpretation.

In [ ]:
example = crc_valid_rows[0]
example_row = {
    "PCI": example["pci"],
    "RNTI": example["grant"]["rnti"],
    "burst subframe": example["burst_sf"],
    "PRBs": f"{example['grant']['prb_tilde0']} + {example['grant']['len_prb']}",
    "MCS": example["grant"]["mcs"],
    "modulation": example["decode"]["mod"],
    "timing (µs)": example["physical"]["lag_us"],
    "DMRS coherence": example["physical"]["coherence"],
    "SNR (dB)": example["decode"]["snr_db"],
    "CRC": example["decode"]["crc"],
    "payload prefix": example["decode"]["payload_hex"][:32] + "…",
}
display(HTML(html_table([example_row])))

## 17. Confirm what was generated and what was reused

The report below came from the executable pipeline cell above. `CREATE` means the notebook generated and stored an artifact during this run. `SKIP` means the expected file already existed and passed its validation check.

In [ ]:
display(HTML(html_table(pipeline_report)))
print("Pipeline report stored at:", RESULTS_DIR / "pipeline_report.json")

# Part 4 — Package validated DL and UL into PCAP

The packaging stage assigns capture-relative timestamps, preserves direction, and gives each `(PCI, RNTI)` pair a distinct MAC-LTE UE context. It combines:

- synthetic MIB records from each decoded PBCH/MIB;
- all decoded DL MAC activity, including broadcast and dedicated traffic;
- only UL transport blocks whose CRC passed.

The PCAPNG timestamps use capture UTC plus segment anchor plus LTE TTI unwrapped from the decoded MIB SFN offset. Timing advance is used to locate and decode the UL waveform; it does not move the final event timestamp to the satellite's arrival time.

In [ ]:
pcap_dir = RESULTS_DIR / "pcap"
pcap_files = [
    pcap_dir / "all_cells_complete_dl_ul_with_mib.pcapng",
    pcap_dir / "all_cells_full_dl_plus_crc_valid_ul.pcap",
    pcap_dir / "all_cells_crc_valid_ul_only.pcap",
]
display(HTML(html_table([pcap_summary(path) for path in pcap_files])))

## 18. Final result and how to inspect it

Open `all_cells_complete_dl_ul_with_mib.pcapng` in Wireshark for the complete trace. Useful display filters include:

```text
mac-lte.direction == 0        # uplink
mac-lte.direction == 1        # downlink
lte-rrc                       # RRC messages such as SIB and connection control
nas-eps                       # NAS signaling when it is visible/decryptable
mac-lte.crc-status == 1       # CRC-valid frames when the field is present
```

For this selected interval, the reference run contains 243 CRC-valid PCI 427 UL transport blocks; valid reruns have recovered 241 or more and may retain an additional CRC-valid adjacent grant. The pipeline therefore enforces a minimum of 241 rather than treating one exact recovery count as a file-integrity invariant. The generated summary reports the actual count.

In [ ]:
final_pcap = pcap_dir / "all_cells_complete_dl_ul_with_mib.pcapng"
print("Final PCAPNG:", final_pcap)
print("Exists:", final_pcap.is_file())
print("Size:", f"{final_pcap.stat().st_size / 1e6:.3f} MB")
print("Wireshark protocol counts:", pcap_protocol_counts(final_pcap).most_common(10))

## 19. What has and has not been proven

- A PCI is accepted only after LTE synchronization and MIB confirmation.
- A SIB1 identity is reported only when Wireshark decodes the SIB1 field from that segment's DL PCAP.
- A bright UL burst is not called a packet by itself.
- DMRS validates the expected PUSCH waveform and improves timing/CFO selection, but does not validate payload bits.
- Only the transport-block CRC promotes a decoded UL payload into the trusted UL PCAP.
- A logical SIB1 node identifier is not automatically a physical satellite/NORAD identifier.
- Encrypted user-plane or protected signaling is preserved as bytes; this tutorial does not claim plaintext where keys are unavailable.